# Flight Price Prediction — XGBoost Versions

Four progressively better XGBoost variants. Each saves a `model_xgb_vN.pkl` and `submission_xgb_vN.csv`.

| Version | Preprocessing | Target | Tuning |
|---------|--------------|--------|--------|
| v1 | Base (ordinal + OHE) | raw | default params |
| v2 | Base | log1p | default params |
| v3 | Base + interactions (route, class_airline, class_x_duration) | log1p | default params |
| v4 | Base + interactions | log1p | Optuna 60 trials |

**Benchmark to beat:** RF tuned $122.60 · GB tuned log $121.92

> **WSL2 note:** `n_jobs=4` (not -1) prevents memory oversubscription when XGBoost parallelism combines with large OHE matrices.

---
## Why XGBoost over sklearn GradientBoostingRegressor

- **Multithreaded** — sklearn GB is single-threaded; XGBoost uses all cores
- **Newton boosting** — second-order gradients give better update steps
- **Richer regularisation** — `reg_alpha` (L1), `reg_lambda` (L2), `gamma` (min gain to split)
- **Column subsampling per tree AND per level** — more decorrelation

## Feature engineering rationale (v3/v4)

- **`route`** (`source→destination`): 18 unique routes each with a distinct price level; reconstructing this from two separate OHE columns needs many splits
- **`class_airline`** (`flight_class_airline`): Business pricing is carrier-specific — Delta Business ≠ Frontier Business
- **`class_x_duration`**: Business fares scale non-linearly with duration; a multiplicative term captures this in one split

In [19]:
import pandas as pd
import numpy as np
import pickle
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_predict

print(f'XGBoost {xgb.__version__}   Optuna {optuna.__version__}')

XGBoost 3.4.1   Optuna 5.0.0


In [20]:
_cwd = Path().resolve()
ROOT   = _cwd if (_cwd / 'data').exists() else _cwd.parent
DATA   = ROOT / 'data'
OUTDIR = ROOT / 'outputs'
OUTDIR.mkdir(exist_ok=True)

train_raw = pd.read_csv(DATA / 'train.csv')
test_raw  = pd.read_csv(DATA / 'test.csv')

train_raw = train_raw[
    (train_raw['price'] > 0) & (train_raw['airline'] != 'Southern_Airways_Express')
].reset_index(drop=True)

KF = KFold(n_splits=5, shuffle=True, random_state=42)

STOPS_MAP = {'zero': 0, 'one': 1, 'two_or_more': 2}
TIME_MAP  = {'Early_Morning': 0, 'Morning': 1, 'Afternoon': 2,
             'Evening': 3, 'Night': 4, 'Late_Night': 5}
CLASS_MAP = {'Economy': 0, 'Business': 1}

print(f'train: {train_raw.shape}   test: {test_raw.shape}')

train: (18050, 12)   test: (4520, 11)


In [21]:
def base_encode(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['stops']          = out['stops'].map(STOPS_MAP)
    out['departure_time'] = out['departure_time'].map(TIME_MAP)
    out['arrival_time']   = out['arrival_time'].map(TIME_MAP)
    out['flight_class']   = out['flight_class'].map(CLASS_MAP)
    return out.drop(columns=[c for c in ['id', 'flight', 'price'] if c in out.columns])

def add_interactions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['route']            = df['source_city'] + '->' + df['destination_city']
    df['class_airline']    = df['flight_class'].astype(str) + '_' + df['airline']
    df['class_x_duration'] = df['flight_class'] * df['duration']
    return df

NUM_BASE = ['duration', 'days_left', 'stops', 'departure_time', 'arrival_time', 'flight_class']
OHE_BASE = ['airline', 'source_city', 'destination_city']
NUM_FE   = NUM_BASE + ['class_x_duration']
OHE_FE   = OHE_BASE + ['route', 'class_airline']

y = train_raw['price'].to_numpy()

X_base = base_encode(train_raw)
X_fe   = add_interactions(X_base)

test_ids    = test_raw['id']
X_test_base = base_encode(test_raw.drop(columns=['id']))
X_test_fe   = add_interactions(X_test_base)

print(f'X_base: {X_base.shape}   X_fe: {X_fe.shape}')
print(f'X_test_base: {X_test_base.shape}   X_test_fe: {X_test_fe.shape}')

X_base: (18050, 9)   X_fe: (18050, 12)
X_test_base: (4520, 9)   X_test_fe: (4520, 12)


In [22]:
def make_pipeline(model, num_cols=NUM_BASE, ohe_cols=OHE_BASE):
    pre = ColumnTransformer([
        ('pass', 'passthrough', num_cols),
        ('ohe',  OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), ohe_cols),
    ])
    return Pipeline([('pre', pre), ('model', model)])

def cv_rmse(pipe, X, y, log_target=False):
    y_fit = np.log1p(y) if log_target else y
    oof   = cross_val_predict(pipe, X, y_fit, cv=KF)
    if log_target:
        oof = np.expm1(oof)
    return float(np.sqrt(np.mean((y - oof) ** 2)))

def save_model_and_submission(pipe, X_train, y_train, X_test, test_ids,
                               tag: str, log_target: bool = False):
    y_fit = np.log1p(y_train) if log_target else y_train
    pipe.fit(X_train, y_fit)
    with open(OUTDIR / f'model_{tag}.pkl', 'wb') as f:
        pickle.dump(pipe, f)
    raw_preds = pipe.predict(X_test)
    if log_target:
        raw_preds = np.expm1(raw_preds)
    preds = np.clip(np.round(raw_preds), 1, None).astype(int)
    sub = pd.DataFrame({'id': test_ids, 'price': preds})
    assert sub.shape == (4520, 2) and sub['price'].isna().sum() == 0
    sub.to_csv(OUTDIR / f'submission_{tag}.csv', index=False)
    print(f'  model  -> model_{tag}.pkl')
    print(f'  submit -> submission_{tag}.csv   price range: {preds.min()}–{preds.max()}')

results = []
print('Helpers ready.')

Helpers ready.


---
## v1 — XGBoost · base preprocessing · raw target

XGBoost baseline with the same feature set used for RF/GB. No tuning.

In [23]:
pipe_v1 = make_pipeline(xgb.XGBRegressor(
    n_estimators=200, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=4, verbosity=0,
))

rmse_v1 = cv_rmse(pipe_v1, X_base, y, log_target=False)
print(f'v1 CV RMSE : ${rmse_v1:.2f}   benchmark: $121.92')

print()
save_model_and_submission(pipe_v1, X_base, y, X_test_base, test_ids, tag='xgb_v1', log_target=False)
results.append({'version': 'v1  base · raw target', 'CV RMSE ($)': round(rmse_v1, 2)})

v1 CV RMSE : $133.95   benchmark: $121.92

  model  -> model_xgb_v1.pkl
  submit -> submission_xgb_v1.csv   price range: 25–1837


---
## v2 — XGBoost · base preprocessing · log target

Same features, fit on `log1p(price)`. Business fare outliers (~$9k) pull leaf means up in raw space;
log space brings their magnitude closer to Economy fares so leaf predictions are more accurate for both classes.

In [24]:
pipe_v2 = make_pipeline(xgb.XGBRegressor(
    n_estimators=200, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=4, verbosity=0,
))

rmse_v2 = cv_rmse(pipe_v2, X_base, y, log_target=True)
print(f'v2 CV RMSE : ${rmse_v2:.2f}   delta vs v1: {rmse_v1 - rmse_v2:+.2f}')

print()
save_model_and_submission(pipe_v2, X_base, y, X_test_base, test_ids, tag='xgb_v2', log_target=True)
results.append({'version': 'v2  base · log target', 'CV RMSE ($)': round(rmse_v2, 2)})

v2 CV RMSE : $128.14   delta vs v1: +5.82

  model  -> model_xgb_v2.pkl
  submit -> submission_xgb_v2.csv   price range: 34–2324


---
## v3 — XGBoost · feature engineering · log target

Adds three interaction features:

- **`route`** — `source_city→destination_city` pair: 18 unique routes each with distinct price level
- **`class_airline`** — `flight_class_airline` combo: 11 values capturing carrier-specific Business premiums
- **`class_x_duration`** — multiplicative: Business fares scale non-linearly with flight length

In [25]:
pipe_v3 = make_pipeline(
    xgb.XGBRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=4, verbosity=0,
    ),
    num_cols=NUM_FE, ohe_cols=OHE_FE,
)

rmse_v3 = cv_rmse(pipe_v3, X_fe, y, log_target=True)
print(f'v3 CV RMSE : ${rmse_v3:.2f}   delta vs v2: {rmse_v2 - rmse_v3:+.2f}')

print()
save_model_and_submission(pipe_v3, X_fe, y, X_test_fe, test_ids, tag='xgb_v3', log_target=True)
results.append({'version': 'v3  FE · log target', 'CV RMSE ($)': round(rmse_v3, 2)})

v3 CV RMSE : $127.57   delta vs v2: +0.56

  model  -> model_xgb_v3.pkl
  submit -> submission_xgb_v3.csv   price range: 36–2089


---
## v4 — XGBoost · feature engineering · log target · Optuna tuning

Optuna **TPE (Tree-structured Parzen Estimator)** — Bayesian optimisation that samples more
from hyperparameter regions that proved good, fewer from bad ones. More efficient than
RandomizedSearchCV at the same number of trials.

**Runtime:** ~10–15 minutes for 60 trials.

In [26]:
def xgb_objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int('n_estimators', 100, 600),
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        max_depth         = trial.suggest_int('max_depth', 3, 8),
        subsample         = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.4, 1.0),
        colsample_bylevel = trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        min_child_weight  = trial.suggest_int('min_child_weight', 1, 30),
        gamma             = trial.suggest_float('gamma', 0.0, 5.0),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        random_state=42, n_jobs=4, verbosity=0,
    )
    pipe = make_pipeline(xgb.XGBRegressor(**params), num_cols=NUM_FE, ohe_cols=OHE_FE)
    return cv_rmse(pipe, X_fe, y, log_target=True)

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(xgb_objective, n_trials=60, show_progress_bar=True)

print(f'\nBest Optuna CV RMSE: ${study.best_value:.2f}')
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

  0%|          | 0/60 [00:00<?, ?it/s]


Best Optuna CV RMSE: $119.65
Best params:
  n_estimators: 435
  learning_rate: 0.2225562832101729
  max_depth: 6
  subsample: 0.8364816969018852
  colsample_bytree: 0.7380804984326842
  colsample_bylevel: 0.5086342592225718
  min_child_weight: 9
  gamma: 0.015972151179408492
  reg_alpha: 2.8610029269687586e-08
  reg_lambda: 3.9436940229099005e-05


In [27]:
best_params = {**study.best_params, 'random_state': 42, 'n_jobs': 4, 'verbosity': 0}

pipe_v4 = make_pipeline(xgb.XGBRegressor(**best_params), num_cols=NUM_FE, ohe_cols=OHE_FE)
rmse_v4 = cv_rmse(pipe_v4, X_fe, y, log_target=True)
print(f'v4 CV RMSE : ${rmse_v4:.2f}   delta vs v3: {rmse_v3 - rmse_v4:+.2f}')
print(f'benchmark  : $121.92         total gain:  {121.92 - rmse_v4:+.2f}')

print()
save_model_and_submission(pipe_v4, X_fe, y, X_test_fe, test_ids, tag='xgb_v4', log_target=True)
results.append({'version': 'v4  FE · log · Optuna', 'CV RMSE ($)': round(rmse_v4, 2)})

with open(OUTDIR / 'xgb_v4_optuna_params.json', 'w') as f:
    json.dump({'best_params': best_params, 'cv_rmse': round(rmse_v4, 2), 'n_trials': 60}, f, indent=2)
print('Saved xgb_v4_optuna_params.json')

v4 CV RMSE : $119.65   delta vs v3: +7.92
benchmark  : $121.92         total gain:  +2.27

  model  -> model_xgb_v4.pkl
  submit -> submission_xgb_v4.csv   price range: 23–2492
Saved xgb_v4_optuna_params.json


---
## Summary

In [28]:
prior = [
    {'version': 'RF  tuned (nb 03)',     'CV RMSE ($)': 122.60},
    {'version': 'GB  tuned log (nb 05)', 'CV RMSE ($)': 121.92},
]

summary = (
    pd.DataFrame(prior + results)
      .sort_values('CV RMSE ($)')
      .reset_index(drop=True)
)
summary.index += 1
print(summary.to_string())

print()
print('Saved files:')
for f in sorted(OUTDIR.glob('*xgb*')):
    print(f'  {f.name}')

                 version  CV RMSE ($)
1  v4  FE · log · Optuna       119.65
2  GB  tuned log (nb 05)       121.92
3      RF  tuned (nb 03)       122.60
4    v3  FE · log target       127.57
5  v2  base · log target       128.14
6  v1  base · raw target       133.95

Saved files:
  model_xgb_smoke.pkl
  model_xgb_smoke_fe.pkl
  model_xgb_smoke_v1.pkl
  model_xgb_smoke_v3.pkl
  model_xgb_v1.pkl
  model_xgb_v2.pkl
  model_xgb_v3.pkl
  model_xgb_v4.pkl
  submission_xgb_smoke.csv
  submission_xgb_smoke_fe.csv
  submission_xgb_smoke_v1.csv
  submission_xgb_smoke_v3.csv
  submission_xgb_v1.csv
  submission_xgb_v2.csv
  submission_xgb_v3.csv
  submission_xgb_v4.csv
  xgb_v4_optuna_params.json
